Step 1: Install, bootstrap instructions

In [ ]:
!pip install --quiet anthropic pydantic
from google.colab import userdata, drive # type: ignore
drive.mount('/content/drive')

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")

from astra_swarm.cassette import cassette
from astra_swarm.alerts import (
    triage_chain, generate_synthetic_alerts,
    parse_alert, summarize, assess_severity,
)

print("ready")

Step 2: Generate 5 synthetic alerts to regenerate them later.

In [ ]:
import json
from pathlib import Path

alerts = []
with cassette("day02_synthetic_alerts", mode="auto"):
    alerts = generate_synthetic_alerts(5)

fixture_path = Path("/content/astra-swarm/data/synthetic/02_alerts.json")
fixture_path.parent.mkdir(parents=True, exist_ok=True)
fixture_path.write_text(json.dumps(alerts, indent=2))

for i, a in enumerate(alerts, 1):
    print(f"--- Alert {i} ---")
    print(a)
    print()

# Use the following to regenerate alerts.
# alerts = json.loads(fixture_path.read_text())

Step 3: Run the chain on all alerts

In [ ]:
import hashlib
for i, alert in enumerate(alerts):
    alert_id = hashlib.md5(alert.encode()).hexdigest()[:8]
    with cassette(f"day02_v1_alert{i:02d}_{alert_id}", mode="auto"):
        raw = alert

        print("=== RAW ===")
        print(raw)
        print()

        parsed = parse_alert(raw)
        print("=== PARSED ===")
        # Day 5 change: Using Pydantic built-in serializer since parse_alert returns ParsedAlert object, not dict.
        # print(json.dumps(parsed, indent=2))
        print(parsed.model_dump_json(indent=2))
        print()

        summary = summarize(parsed)
        print("=== SUMMARY ===")
        print(summary)
        print()

        verdict = assess_severity(parsed, summary)
        print("=== VERDICT ===")
        # Day 5 change: Using Pydantic built-in serializer since parse_alert returns SeverityVerdict object, not dict.
        # print(json.dumps(verdict, indent=2))
        print(verdict.model_dump_json(indent=2))

Step 4: Run the whole chain on all 5 alerts

In [ ]:
results = None
with cassette("day02_wholechain_v1", mode = "auto"):
    results = [triage_chain(a) for a in alerts]

print(f"{'#':<3} {'severity':<10} {'conf':<6} description")
print("-" * 80)
for i, r in enumerate(results, 1):
    v = r.verdict
    desc = r.parsed.description
    print(f"{i:<3} {v.severity.value:<10} {v.confidence:<6.2f} {desc}")

Temporary debug cells

In [ ]:
# Google Drive mounting and environment variable checks

import os
from pathlib import Path

# 1. Is Drive actually mounted?
mounted = Path("/content/drive/MyDrive").exists()
print(f"1. Drive mounted at /content/drive/MyDrive: {mounted}")

# 2. What did the env var actually get set to?
env = os.environ.get("ASTRA_CASSETTE_DIR", "(not set)")
print(f"2. ASTRA_CASSETTE_DIR env: {env}")

# 3. What did the cassette module ACTUALLY resolve to?
from astra_swarm.cassette import _cassette_dir
print(f"3. cassette module dir: {_cassette_dir()}")

# 4. Does that directory exist? Any files?
if _cassette_dir().exists():
    files = list(_cassette_dir().iterdir())
    print(f"4. Directory exists. Files: {[f.name for f in files]}")
else:
    print(f"4. Directory does NOT exist: {_cassette_dir()}")

# 5. Any cassette pickle in the LEGACY location?
legacy = Path("/content/astra-swarm/cassettes")
if legacy.exists():
    files = list(legacy.iterdir())
    print(f"5. Legacy /content path files: {[f.name for f in files]}")
else:
    print(f"5. Legacy /content path does NOT exist.")

In [ ]:
# Diagnostic: is the cassette even monkey-patching the client?

import inspect
from anthropic import Anthropic
from anthropic.resources.messages import Messages

# Before entering the cassette block, capture the original method
before = Messages.create
print(f"Before cassette: create method file = {inspect.getfile(before)}")
print(f"Before cassette: create method qualname = {before.__qualname__}")

from astra_swarm.cassette import cassette

with cassette("diagnostic_test", mode="auto"):
    during = Messages.create
    print(f"\nInside cassette: create method file = {inspect.getfile(during)}")
    print(f"Inside cassette: create method qualname = {during.__qualname__}")
    print(f"Same method as before? {before is during}")

Final Step: unmount & cleanup

In [ ]:
drive.flush_and_unmount()